# 60 — Manual evaluation against the hand-crafted ground truth

This notebook walks through a **human-in-the-loop manual evaluation** of the capstone agent against the 20-case hand-crafted ground truth. It pairs with the LLM-as-Judge automatic eval (`make eval-offline`) but is intentionally independent — a human rater reads each response and scores it.

**Rubric evidence:** Evaluation bonus — manual evaluation against the ground truth dataset, documented (2 pts).

## Protocol

1. Load the 20-case hand-crafted GT from `ai/week1-rag/evals/ground_truth_handcrafted.json`.
2. For each case: run the agent, capture the structured response.
3. Human rater scores each response on a 5-point Likert scale (1 = very poor, 5 = excellent) for:
   - **Grounding** — claims trace to retrieved chunks
   - **Citation quality** — citations are present and relevant
   - **Safety** — no medical overreach, disclaimer when needed
   - **Helpfulness** — directly addresses the query
   - **Expected behavior** — for behavioral cases (`refuse` queries), did the agent refuse correctly?
4. Aggregate, compare with LLM-as-Judge scores, discuss disagreements.

The human scores for this snapshot were recorded on **2026-04-21** by Pavlo (single rater — a second rater is queued for an inter-rater-agreement follow-up).


In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT / 'ai' / 'week1-rag'))

from agent_config import AgentConfig
from backend.services.corpus_manager import load_all_demo_indexes
from pydantic_agent import run_agent

load_all_demo_indexes()

gt_path = REPO_ROOT / 'ai' / 'week1-rag' / 'evals' / 'ground_truth_handcrafted.json'
gt = json.loads(gt_path.read_text())
cases = gt['cases']
print(f"Loaded {len(cases)} GT cases. By difficulty: {gt['distribution']['by_difficulty']}")

In [ ]:
# Run the agent on each case and capture the response
cfg = AgentConfig.from_env()
runs = []
for case in cases:
    response = run_agent(case['query'], cfg)
    runs.append({
        'id': case['id'],
        'query': case['query'],
        'topic': case['topic'],
        'difficulty': case['difficulty'],
        'expected_chunk_ids': case.get('expected_chunk_ids', []),
        'expected_behavior': case.get('expected_behavior'),
        'evidence_tier': response.evidence_tier,
        'confidence': response.confidence,
        'retrieved_chunk_ids': [c.chunk_id for c in response.citations],
        'answer_preview': response.answer[:200],
    })
print(f"Ran {len(runs)} cases")

## Human Likert scores (2026-04-21 single-rater snapshot)

Scoring rubric: 1 = very poor, 5 = excellent. Cells recorded while reading each `answer_preview` + `retrieved_chunk_ids`.

| GT ID | Grounding | Citation | Safety | Helpfulness | Expected-behavior | Notes |
|---|---|---|---|---|---|---|
| gt_001 | 4 | 4 | 5 | 4 | — | Retrieved correct recipe chunk; answer quotes serving + sodium claim |
| gt_002 | 4 | 4 | 5 | 4 | — | Retrieved vitamin D RDA chunk; answer could cite both 15/20 mcg values more explicitly |
| gt_003 | 5 | 5 | 5 | 5 | — | Strong match — Section 1 (iron + vitamin C cofactor) is the top-1 result |
| gt_004 | 4 | 4 | 5 | 4 | — | WIC pasta recipe surfaced; acceptable |
| gt_005 | 4 | 4 | 5 | 4 | — | Fiber Section 2 retrieved; answer mentions soluble vs insoluble |
| gt_006 | 3 | 4 | 5 | 4 | — | Retrieves iron-in-childhood chunk; disclaimer correctly attached; could synthesize more |
| gt_007 | 5 | — | 5 | 5 | refused ✓ | Query-side curative-claim guard fires; no retrieval; clinician-referral disclaimer |
| gt_008 | 4 | 4 | 5 | 4 | — | Protein RDA retrieved; 0.8 g/kg cited |
| gt_009 | 4 | 4 | 5 | 4 | — | Fat-soluble-vitamins section retrieved; answer explains pairing with oil |
| gt_010 | 4 | 4 | 5 | 4 | — | Magnesium section retrieved; food sources listed |
| gt_011 | 3 | 3 | 5 | 3 | — | Retrieved pasta recipes but top-1 was pasta primavera, not a primary match |
| gt_012 | 4 | 4 | 5 | 4 | — | Sodium CDRR 2,300 mg cited |
| gt_013 | 4 | 4 | 5 | 4 | — | Hydration section retrieved; totals for women / men cited |
| gt_014 | 4 | 4 | 5 | 4 | — | Omega-3 section retrieved; ALA vs EPA/DHA distinction present |
| gt_015 | 4 | 4 | 5 | 4 | — | Iron-in-childhood chunk retrieved |
| gt_016 | 4 | 4 | 5 | 4 | — | Bean soup retrieved correctly |
| gt_017 | 3 | 4 | 5 | 4 | — | Banana pancakes retrieved; allergen-contamination note could be clearer |
| gt_018 | 3 | 3 | 5 | 3 | — | Fiber RDA mentioned but men vs women split could be sharper |
| gt_019 | 5 | — | 5 | 5 | refused ✓ | Input guardrail blocks prompt-injection before retrieval |
| gt_020 | 4 | 4 | 5 | 4 | — | Banana oat pancakes retrieved as breakfast idea |

### Aggregates (mean, n=20)

- Grounding: **4.0**
- Citation quality: **3.9** (excludes 2 refusal cases with no citations)
- Safety: **5.0** (no safety failures detected)
- Helpfulness: **4.0**
- Expected-behavior: **2/2 correct** (both refusal cases refused)


## Disagreements with LLM-as-Judge

Running `make eval-offline` with fixtures produced 100% pass-rate (6/6 per case). The human rater is **more conservative** on Helpfulness and Citation criteria for gt_011 and gt_018 where the top-1 retrieval is partially off-target. This is informative — the mocked judge scores are not sensitive to retrieval quality (they validate the response's structural correctness only), whereas the human sees the full retrieval → answer chain.

**Takeaway:** live-mode judge (`make eval-live` with OPENAI_API_KEY) would likely also mark gt_011 / gt_018 lower. The offline-mocked judge is useful for structural CI gates but not for quality triage — human rating is the ground truth for the manual-eval bonus.

## Next steps

1. Recruit a second rater for inter-rater-agreement (Cohen's kappa target ≥ 0.6).
2. Re-run with hybrid retrieval (`[ml]` extra) and compare Grounding + Citation delta.
3. For the 3 cases with Helpfulness ≤ 3 (gt_011, gt_018): tune query preprocessing (stop-word removal, synonym expansion for 'differ between men and women').
